# Routing, Memory & Tools
_Kapruka Gift Agent — Foundation Concepts_

---

This notebook covers the Kapruka gift logistics pipeline, the 4 routing paths, and the 4-layer memory system. It is the prerequisite for Notebook 02 (LangGraph multi-agent system).

## What You Will Learn

| Section | Topic |
|---------|-------|
| **1** | Build the agent — wiring all components together |
| **2** | User identification from free-form chat |
| **3** | Routing engine — 4 routes: `direct`, `crm`, `rag`, `web_search` |
| **4** | Short-term memory — conversation ring buffer |
| **5** | Memory distillation — LLM extracts long-term facts |
| **6** | Long-term semantic recall — pgvector cosine search |
| **7** | Episodic memory — full conversation snapshots |
| **8** | Procedural memory — workflow guidance |
| **9** | Without vs. with memory — the impact of context |
| **10** | Multi-turn progressive conversation |
| **11** | LangFuse observability |

## Architecture (Week 07 — Imperative Pipeline)

```
User Message
|
AgentOrchestrator.chat()
  1. _recall_memory()   -> ST turns + LT facts   -> memory_context (str)
  2. router.route()     -> RouteDecision (crm | rag | web_search | direct)
  3. _dispatch()        -> CRMTool | RAGTool | WebSearchTool -> tool_output (str)
  4. _synthesise()      -> LLM merges memory + tool output -> final answer
  5. _store_turns()     -> ST write (Supabase ring buffer)
  6. _maybe_distill()   -> LLM extracts LT facts (if triggered)
```

## Prerequisites
```bash
make seed-crm-xl        # Populate CRM with users, orders, logistics data
make ingest-qdrant      # Ingest knowledge base into Qdrant (auto-runs on first start)
python scripts/seed_procedures.py  # Load procedural memory
```

---
## Section 0 Ã¢â‚¬â€ Setup

In [ ]:
import sys, os, re, time, json
sys.path.insert(0, "../src")

from dotenv import load_dotenv
load_dotenv()

from infrastructure.log import setup_logging
from loguru import logger
setup_logging("INFO", for_notebook=True)

import pandas as pd
from datetime import datetime
from sqlalchemy.orm import sessionmaker

from infrastructure.db import create_tables, get_sql_engine
from infrastructure.db.crm_models import User, DeliveryZone, DeliverySlot, CourierProfile, ProductDeliveryRule, DeliveryHistory
from infrastructure.llm import get_chat_llm, get_default_embeddings
from memory import (
    ConversationTurn, ShortTermMemoryStore, LongTermMemoryStore,
    EpisodicMemoryStore, ProceduralMemoryStore,
    MemoryDistiller, MemoryRecaller, create_episode_from_turns,
)
from infrastructure.config import ST_MAX_TURNS, ST_TTL_SECONDS

create_tables()
embedder   = get_default_embeddings()
llm        = get_chat_llm()
st_store   = ShortTermMemoryStore()
lt_store   = LongTermMemoryStore(embedder)
distiller  = MemoryDistiller(llm, lt_store)
recaller   = MemoryRecaller(st_store, lt_store)

logger.success("Environment ready")
logger.info(f"  LLM       : {getattr(llm, 'model_name', 'unknown')}")
logger.info(f"  ST store  : Supabase (st_turns)")
logger.info(f"  LT store  : Supabase pgvector")

---
## Section 1 — Build the Agent

`build_agent()` wires all components:
- **3 LLMs**: Router (`gpt-4o-mini`) · Extractor (`llama-3.1-8b-instant`) · Chat (`gemini-2.5-flash`)
- **3 Tools**: CRM (SQLAlchemy -> Supabase) · RAG (CAG -> CRAG -> Qdrant) · Web (Tavily)
- **Memory**: ShortTermStore + LongTermMemoryStore + MemoryDistiller + MemoryRecaller
- **Observability**: LangFuse v3 traces every step

On first run, `ensure_kb_ingested()` checks if the Qdrant knowledge collection has data — if not, it runs the ingestion pipeline automatically.

In [ ]:
from agents import build_agent, AgentResponse

agent = build_agent(enable_crm=True, enable_rag=True, enable_web=True)

logger.success("Agent built")
logger.info(f"  CRM tool   : {'Ã¢Å“â€œ' if agent.crm_tool else 'Ã¢Å“â€”'}")
logger.info(f"  RAG tool   : {'Ã¢Å“â€œ' if agent.rag_tool else 'Ã¢Å“â€”'}")
logger.info(f"  Web search : {'Ã¢Å“â€œ' if agent.web_tool else 'Ã¢Å“â€”'}")

# Ã¢â€â‚¬Ã¢â€â‚¬ Visualize the LangGraph StateGraph Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
from IPython.display import Image, display

graph_png = agent.graph.get_graph(xray=True).draw_mermaid_png()
display(Image(graph_png))

---
## Section 2 — User Identification from Chat

In production, Kapruka users interact through WhatsApp or web chat. Their identity is inferred from the conversation itself — not from a hardcoded variable. The `extract_phone()` helper normalises Sri Lankan phone numbers:

| User types | Normalised to |
|---|---|
| `+94 78 10 30 736` | `94781030736` |
| `0781030736` | `94781030736` |
| `+94-781-030-736` | `94781030736` |
| `(078) 103-0736` | `94781030736` |

In [ ]:
def extract_phone(text: str) -> str:
    """Extract and normalise a Sri Lankan phone number from free-form text."""
    match = re.search(r"\+?\d[\d\s\-\.\(\)]{7,18}\d", text)
    if not match:
        raise ValueError("No phone number found in the message.")
    raw = re.sub(r"\D", "", match.group())
    if raw.startswith("0") and len(raw) == 10:
        raw = "94" + raw[1:]
    elif len(raw) == 9 and not raw.startswith("94"):
        raw = "94" + raw
    elif raw.startswith("94") and len(raw) == 11:
        pass
    else:
        raise ValueError(
            f"Unsupported phone format after cleanup: {raw}. Expected 9-12 digits."
        )
    logger.info(f"  Normalised - {raw}")
    return raw


def show_response(resp: AgentResponse, label: str = "") -> None:
    """Pretty-print an AgentResponse."""
    print("=" * 72)
    if label:
        print(f"  {label}")
    print("=" * 72)
    print(f"  Route   : {resp.route}" + (f" / {resp.action}" if resp.action else ""))
    print(f"  Latency : {resp.latency_ms} ms")
    if resp.memory_context and resp.memory_context.strip():
        lines = resp.memory_context.strip().split("\n")
        print(f"\n  Memory context ({len(lines)} lines):")
        for ln in lines[:8]:
            print(f"    {ln}")
        if len(lines) > 8:
            print(f"    ... ({len(lines) - 8} more lines)")
    if resp.tool_output and resp.tool_output.strip():
        print(f"\n  Tool output:")
        for ln in resp.tool_output.strip().split("\n")[:6]:
            print(f"    {ln}")
    print(f"\n  Answer:\n    {resp.answer}")
    print("=" * 72)


def _crm_session():
    return sessionmaker(bind=get_sql_engine())()


def show_user(external_user_id: str) -> None:
    session = _crm_session()
    try:
        user = session.query(User).filter(User.external_user_id == external_user_id).first()
        if not user:
            logger.warning(f"No CRM user found for external_user_id={external_user_id}")
            return

        rows = [
            {"Field": "User ID", "Value": user.user_id or "-"},
            {"Field": "External ID", "Value": user.external_user_id or "-"},
            {"Field": "Name", "Value": user.full_name or "-"},
            {"Field": "Phone", "Value": user.phone or "-"},
            {"Field": "Email", "Value": user.email or "-"},
            {"Field": "District", "Value": user.district or "-"},
            {"Field": "Province", "Value": user.province or "-"},
            {"Field": "Address", "Value": user.address or "-"},
            {"Field": "Notes", "Value": user.notes or "-"},
            {"Field": "Active", "Value": "Yes" if user.active else "No"},
        ]

        print("CRM User Record (Supabase users table)")
        display(pd.DataFrame(rows).style.hide(axis="index"))
    finally:
        session.close()


def resolve_user_id(external_user_id: str) -> str:
    session = _crm_session()
    try:
        user = session.query(User).filter(User.external_user_id == external_user_id).first()
        if not user:
            raise ValueError(
                f"No CRM user found for external_user_id={external_user_id}."
            )
        return user.user_id
    finally:
        session.close()


# Ã¢â€â‚¬Ã¢â€â‚¬ Identify user Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
greeting = "Hi there! I'm Thilanka, and my mobile number is +94771234567."
EXTERNAL_USER_ID = extract_phone(greeting)
USER_ID = resolve_user_id(EXTERNAL_USER_ID)
SESSION_ID = "nb01-demo"

logger.success(f"EXTERNAL_USER_ID  = {EXTERNAL_USER_ID}")
logger.success(f"USER_ID           = {USER_ID}")
logger.info(f"SESSION  = {SESSION_ID}")
show_user(EXTERNAL_USER_ID)

---
## Section 3 Ã¢â‚¬â€ The Routing Engine

The **QueryRouter** sends the user message + memory context to `gpt-4o-mini` and receives a structured `RouteDecision`:

```python
@dataclass
class RouteDecision:
    route:      str    # crm | rag | web_search | direct
    confidence: float  # 0.0 Ã¢â‚¬â€œ 1.0
    reasoning:  str
    action:     Optional[str]       # e.g. lookup_patient, search_doctors
    params:     Dict[str, Any]      # extracted parameters for the tool
```

Each route dispatches to a different tool:

| Route | Tool | Backend |
|---|---|---|
| `direct` | None Ã¢â‚¬â€ LLM answers directly | Ã¢â‚¬â€ |
| `crm` | `CRMTool` | SQLAlchemy Ã¢â€ â€™ Supabase PostgreSQL |
| `rag` | `RAGTool` | CAG cache (Qdrant) Ã¢â€ â€™ CRAG Ã¢â€ â€™ Qdrant KB |
| `web_search` | `WebSearchTool` | Tavily API |

### 3a Ã‚Â· `direct` Route Ã¢â‚¬â€ Greeting (No Tool)
Conversational queries are answered by the LLM directly. No tool is dispatched. Memory context (if any) is still injected.

In [ ]:
resp = agent.chat(user_message=greeting, user_id=USER_ID, session_id=SESSION_ID)
show_response(resp, "DIRECT route Ã¢â‚¬â€ greeting")

### 3b - "crm" Route - Delivery Feasibility

Delivery-feasibility and logistics queries route to the CRM tool, which reads the structured logistics tables in Supabase PostgreSQL via SQLAlchemy.

In [ ]:
resp = agent.chat(
    user_message="Can you check same-day delivery availability in Kandy for a cake? My phone number is 077 123 4567.",
    user_id=USER_ID, session_id=SESSION_ID,
)
show_response(resp, "CRM route - check_delivery_coverage")

# Verify against the underlying logistics tables
session = _crm_session()
try:
    zone = session.query(DeliveryZone).filter(DeliveryZone.district == "Kandy").first()
    rule = session.query(ProductDeliveryRule).filter(ProductDeliveryRule.product_type == "cake").first()
    slots = (
        session.query(DeliverySlot)
        .filter(DeliverySlot.district == "Kandy")
        .order_by(DeliverySlot.slot.asc())
        .all()
    )

    if zone:
        zone_rows = [{
            "District": zone.district,
            "Delivery Available": zone.delivery_available,
            "Same Day": zone.same_day,
            "Express": zone.express_available,
            "Min Notice Hours": zone.minimum_notice_hours,
            "Active Couriers": zone.active_couriers,
        }]
        print("\nDB verification (Supabase -> delivery_zones):")
        display(pd.DataFrame(zone_rows))

    if rule:
        rule_rows = [{
            "Product Type": rule.product_type,
            "Fragile": rule.fragile,
            "Same Day Allowed": rule.same_day_allowed,
            "Min Notice Hours": rule.minimum_notice_hours,
            "Max Distance KM": rule.max_delivery_distance_km,
        }]
        print("\nDB verification (Supabase -> product_delivery_rules):")
        display(pd.DataFrame(rule_rows))

    if slots:
        slot_rows = [{
            "District": s.district,
            "Slot": s.slot,
            "Capacity": s.capacity,
            "Available": s.available,
        } for s in slots[:5]]
        print("\nDB verification (Supabase -> delivery_slots):")
        display(pd.DataFrame(slot_rows))
finally:
    session.close()

### 3c - "crm" Route - Courier Availability

Courier-availability queries also route to CRM. The router extracts the district and dispatches to the structured logistics lookup path.

In [ ]:
resp = agent.chat(
    user_message="Which couriers are available in Colombo for deliveries today?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show_response(resp, "CRM route - search_couriers")

# Verify against actual DB rows
session = _crm_session()
try:
    couriers = (
        session.query(CourierProfile)
        .filter(CourierProfile.district == "Colombo", CourierProfile.availability == True)
        .order_by(CourierProfile.rating.desc())
        .limit(5)
        .all()
    )
    if couriers:
        df = pd.DataFrame([
            {
                "Courier": c.name,
                "District": c.district,
                "Vehicle": c.vehicle_type,
                "Available": c.availability,
                "Max/Day": c.max_deliveries_per_day,
                "Rating": c.rating,
            }
            for c in couriers
        ])
        print("\nAvailable couriers in DB (Supabase -> courier_profiles):")
        display(df)
finally:
    session.close()

### 3d - "rag" Route - Product Catalog and Internal Knowledge

Catalog questions and internal Kapruka policy questions route to the RAG pipeline.

Pipeline overview:
- Query -> CAG cache lookup in Qdrant
- On cache miss -> CRAG retrieval and confidence gating
- If confidence is low -> broaden retrieval
- Generate answer -> cache result for future semantic hits

The CAG cache is pre-warmed with FAQ content from config/faqs.yaml, so repeated internal questions can be answered instantly.

In [ ]:
# First call -> cache MISS -> internal FAQ / catalog retrieval -> cached
resp = agent.chat(
    user_message="How does Kapruka validate delivery feasibility before confirming a gift recommendation?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show_response(resp, "RAG route - internal delivery knowledge")

In [ ]:
# Second call -> similar semantic intent -> should benefit from the CAG cache
resp2 = agent.chat(
    user_message="What checks do you perform on delivery location and product restrictions before confirming gifts?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show_response(resp2, "RAG route - CAG cache / semantic reuse")

# Inspect cache stats
stats = agent.rag_tool.cache_stats()
print(f"\nCAG Cache stats:")
print(f"  Entries    : {stats['total_cached']}")
print(f"  Threshold  : {stats['similarity_threshold']}")
print(f"  Backend    : {stats['backend']}")
print(f"  Collection : {stats['collection']}")
print(f"  TTL        : {stats['ttl_seconds']}s")
print(f"  Available  : {stats['available']}")

### 3e - "web_search" Route - Real-Time Delivery Impact

For questions needing live external data, such as weather or traffic conditions that may affect deliveries today, the router dispatches to Tavily Web Search.

In [ ]:
resp = agent.chat(
    user_message="Is heavy rain or traffic affecting Colombo deliveries today?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show_response(resp, "WEB SEARCH route - live delivery-impact lookup")

---
## Section 4 Ã¢â‚¬â€ Memory System: 4 Layers

The agent has 4 memory types working together:

| Layer | What It Stores | Retrieval | Backend |
|---|---|---|---|
| **Short-Term** | Recent conversation turns | Last N turns (recency) | Supabase `st_turns` |
| **Long-Term Semantic** | Distilled facts & preferences | Cosine similarity (pgvector) | Supabase `mem_facts` |
| **Episodic** | Full conversation sessions | Cosine similarity on summary | Supabase `mem_episodes` |
| **Procedural** | Step-by-step workflows | Cosine similarity | Supabase `mem_procedures` |

### 4a Ã‚Â· Short-Term Memory Ã¢â‚¬â€ Conversation Ring Buffer

- **Capacity**: Ring buffer Ã¢â‚¬â€ last `ST_MAX_TURNS=30` turns per user/session
- **TTL**: 24 hours (configurable)
- **Purpose**: Conversational continuity within and across sessions

In [ ]:
MEM_USER    = USER_ID
MEM_SESSION = "nb01-memory-demo"

# Build a realistic Kapruka shopping-preference conversation
conversation = [
    ("user",      "Hi, I'm Thilanka. I usually send gifts to my sister in Kandy."),
    ("assistant", "Noted. I can keep your Kandy delivery context in mind for future recommendations."),
    ("user",      "From now on, remember that my usual budget is Rs. 5,000 to Rs. 8,000."),
    ("assistant", "Got it. I'll remember your normal gift budget range."),
    ("user",      "She loves chocolates and flowers but dislikes perfumes."),
    ("assistant", "Understood. I'll prioritize chocolates and flowers and avoid perfumes."),
    ("user",      "Please always remember that she has a peanut allergy."),
    ("assistant", "Noted. I'll treat peanut-containing gifts as unsuitable."),
]

print(f"Storing {len(conversation)} turns in short-term memory...\n")
for role, content in conversation:
    turn = ConversationTurn(
        user_id=MEM_USER, session_id=MEM_SESSION,
        role=role, content=content, ts=time.time(),
    )
    st_store.append(turn, ST_MAX_TURNS, ST_TTL_SECONDS)
    emoji = "USER" if role == "user" else "BOT"
    print(f"  {emoji:4s} [{role:9s}]: {content}")
    time.sleep(0.05)

logger.success(f"\n{len(conversation)} turns stored in Supabase (st_turns)")

In [ ]:
recent = st_store.recent(MEM_USER, MEM_SESSION, k=10)

print(f"Retrieved {len(recent)} turns from ST memory:\n")
for i, t in enumerate(recent, 1):
    emoji = "Ã°Å¸â€˜Â¤" if t.role == "user" else "Ã°Å¸Â¤â€“"
    age = f"{time.time() - t.ts:.0f}s ago"
    print(f"  {i}. {emoji} [{t.role:9s}] ({age}): {t.content}")

---
## Section 5 - Memory Distillation: LLM Extracts Long-Term Facts

The MemoryDistiller watches the conversation and extracts important facts when:
- Turn count >= 5, OR
- Conversation contains keywords such as "remember", "from now on", "remind me", "always", or "never"

Process:
- Conversation turns
- LLM memory extractor
- JSON array of extracted facts and tags
- score_memory_fact()
- dedupe_facts()
- LongTermMemoryStore.upsert() into pgvector-backed memory

In [ ]:
should = distiller.should_distill(recent)

print(f"Should distill? {should}")
print(f"\nTrigger analysis:")
print(f"  Turn count    : {len(recent)} (threshold: 5)")
for kw in ["remember", "from now on", "remind me", "always", "never"]:
    hit = any(kw in t.content.lower() for t in recent)
    print(f"  '{kw}'  : {hit}")

In [ ]:
print("Running memory distillation (LLM call)...\n")
facts = distiller.distill(MEM_USER, recent)

logger.success(f"Extracted {len(facts)} long-term facts:")
for i, f in enumerate(facts, 1):
    print(f"\n  {i}. {f.text}")
    print(f"     Score : {f.score:.3f}")
    print(f"     Tags  : {', '.join(f.tags)}")

In [ ]:
# Verify facts are semantically searchable via pgvector
test_queries = ["gift budget", "peanut allergy", "Kandy delivery preferences"]

print("Semantic search over long-term memory (pgvector cosine):")
for q in test_queries:
    results = lt_store.query(MEM_USER, q, k=3, threshold=0.3)
    print(f"\n  Query: '{q}'  ->  {len(results)} fact(s)")
    for r in results:
        print(f"    -> {r.text}  [score={r.score:.3f}]")

---
## Section 6 Ã¢â‚¬â€ Long-Term Semantic Recall: Token-Budgeted Retrieval

The **MemoryRecaller** combines ST and LT memories within a **token budget**:

```
max_tokens = 500
  Ã¢â€Å“Ã¢â€â‚¬ Short-term budget : 60% = 300 tokens  (recent turns, max 4)
  Ã¢â€â€Ã¢â€â‚¬ Long-term budget  : 40% = 200 tokens  (semantic facts, max 3)
```

LT facts are retrieved via **cosine similarity** against the user's query Ã¢â‚¬â€ so the most *relevant* facts surface, not just the most recent.

In [ ]:
query = "What gift preferences, budget, and delivery constraints should I consider for Thilanka's sister in Kandy?"

st_turns, lt_facts = recaller.recall(
    user_id=MEM_USER, session_id=MEM_SESSION,
    query=query, k_st=6, k_lt=5, max_tokens=500,
)

print(f"Recalled: {len(st_turns)} ST turns  +  {len(lt_facts)} LT facts")

st_tokens = sum(recaller.count_tokens(t.content) for t in st_turns)
lt_tokens = sum(recaller.count_tokens(f.text)    for f in lt_facts)
used      = st_tokens + lt_tokens
denom     = used if used > 0 else 1

print(f"\nToken budget:")
print(f"  Total   : {used}/500")
print(f"  ST (60%): {st_tokens} tokens ({st_tokens/denom*100:.1f}% of used)")
print(f"  LT (40%): {lt_tokens} tokens ({lt_tokens/denom*100:.1f}% of used)")

st_context = recaller.format_context(st_turns)
if lt_facts:
    lt_lines = ["=== REMEMBERED FACTS ==="]
    for i, f in enumerate(lt_facts, 1):
        lt_lines.append(f"{i}. {f.text}  [{', '.join(f.tags)}]")
    lt_context = "\n".join(lt_lines)
else:
    lt_context = ""

full_context = "\n".join(part for part in [st_context, lt_context] if part)

print(f"\nMemory context passed to LLM:")
print("-" * 60)
print(full_context)

---
## Section 7 - Episodic Memory: Full Conversation Snapshots

Episodic memory stores complete sessions, not just extracted facts.

- Semantic vs episodic: "The customer usually buys chocolates and flowers" is a semantic fact; the full 8-turn conversation where those preferences were shared is an episode.
- Use case: "What did we discuss about gifts for the sister in Kandy?" -> episodic search returns the session.

In [ ]:
episodic_store = EpisodicMemoryStore(embedder)

# Create episode from the conversation turns (LLM generates summary)
episode = create_episode_from_turns(
    user_id=MEM_USER, session_id=MEM_SESSION,
    turns=recent, llm=llm,
)

print(f"Episode created:")
print(f"  Session : {episode.session_id}")
print(f"  Turns   : {episode.turn_count}")
print(f"  Topics  : {', '.join(episode.topic_tags)}")
print(f"  Summary : {episode.summary}")

episodic_store.store_episode(episode)
logger.success("Episode stored in Supabase pgvector")

# Query episodic memory
for q in ["gift preference discussion", "delivery to Kandy"]:
    eps = episodic_store.query_episodes(MEM_USER, q, k=2, threshold=0.3)
    print(f"\n  Query '{q}' -> {len(eps)} episode(s)")
    for ep in eps:
        print(f"    Topics: {', '.join(ep.topic_tags)} | {ep.turn_count} turns")
        print(f"    Summary: {ep.summary[:120]}...")

---
## Section 8 - Procedural Memory: Workflow Guidance

Procedural memory stores step-by-step workflows that help the assistant handle recurring Kapruka tasks consistently.

Examples:
- updating customer preferences
- product search and recommendation flow
- delivery and logistics validation
- reflection checks before finalizing recommendations

In [ ]:
from memory.prompts import format_procedures

proc_store = ProceduralMemoryStore()
query = "Check same-day delivery in Kandy for a cake and explain the logistics steps"

procedures = proc_store.query_procedures(query, top_k=2, threshold=0.2)
if not procedures:
    procedures = proc_store.list_all_procedures()[:3]
    if procedures:
        logger.warning("No close semantic match found; showing available procedures instead.")

if procedures:
    for p in procedures:
        similarity = getattr(p, "similarity", None)
        if similarity is not None:
            logger.success(f"Matched: {p.name}  (similarity={similarity:.2f})")
        else:
            logger.success(f"Available: {p.name}")
        print(f"  Category    : {p.category}")
        print(f"  Description : {p.description}")
        print(f"  Steps       : {len(p.steps)}")
    print("\nFormatted for agent prompt:")
    print("=" * 60)
    print(format_procedures(procedures))
else:
    logger.warning("No procedures found in mem_procedures. Verify your Supabase schema and seed data.")

---
## Section 9 - Without vs. With Memory: The Impact of Context

This is the core demonstration of why memory matters. The same LLM gives:
- a generic answer without memory
- a personalized answer with recalled context

Same model. Same query. Different context -> fundamentally different response.

In [ ]:
compare_query = "What kind of gifts does Thilanka usually prefer for his sister in Kandy, and what budget should I target?"

# Without memory
print("=" * 72)
print("WITHOUT memory (raw LLM - no context):")
print("=" * 72)
baseline = llm.invoke(compare_query)
baseline_text = baseline.content if hasattr(baseline, "content") else str(baseline)
print(baseline_text)

# Rebuild full_context if this cell is run independently
st_ctx = recaller.format_context(st_turns)
if lt_facts:
    lt_lines = ["=== REMEMBERED FACTS ==="]
    for i, f in enumerate(lt_facts, 1):
        lt_lines.append(f"{i}. {f.text}  [{', '.join(f.tags)}]")
    lt_ctx = "\n".join(lt_lines)
else:
    lt_ctx = ""
full_context = "\n".join(part for part in [st_ctx, lt_ctx] if part)
injected_tokens = used

prompt = f"{full_context}\n\nUSER QUERY: {compare_query}\n\nAnswer based on the above:"

print("\n" + "=" * 72)
print(f"WITH memory ({injected_tokens} tokens of context injected):")
print("=" * 72)
memory_resp = llm.invoke(prompt)
memory_text = memory_resp.content if hasattr(memory_resp, "content") else str(memory_resp)
print(memory_text)

print("\n" + "=" * 72)
logger.error("WITHOUT memory: the answer is generic because the model lacks user context")
logger.success(f"WITH memory ({injected_tokens} tokens): the answer can use the stored budget, preferences, and delivery context")

---
## Section 10 â€” Multi-Turn: Progressive Memory Building

Watch memory accumulate across turns. Each turn adds to short-term memory, and important facts are distilled into long-term memory so they survive across sessions.

In [ ]:
MULTI_SESSION = "nb01-multiturn"
first_msg = "Hi, I'm Thilanka. My mobile is 077 123 4567. I usually send gifts to my sister in Kandy."
MULTI_USER = extract_phone(first_msg)

messages = [
    first_msg,
    "Please remember that my usual budget is Rs. 5,000 to Rs. 8,000.",
    "My sister loves chocolates and flowers but dislikes perfumes.",
    "Can you check whether same-day delivery is available in Kandy for chocolates?",
    "Recommend a birthday gift under Rs. 6,000 for her.",
    "What do you remember about my gift preferences and delivery needs?",
]

print("Multi-Turn Conversation - Progressive Memory Building")
print("=" * 72)

for i, msg in enumerate(messages, 1):
    print(f"\n{'-' * 72}")
    print(f"Turn {i}: {msg}")
    print("-" * 72)

    resp = agent.chat(user_message=msg, user_id=MULTI_USER, session_id=MULTI_SESSION)

    ctx_lines = len(resp.memory_context.strip().split("\n")) if resp.memory_context.strip() else 0

    print(f"  Route          : {resp.route}" + (f" / {resp.action}" if resp.action else ""))
    print(f"  Memory context : {ctx_lines} lines (grows each turn)")
    print(f"  Latency        : {resp.latency_ms}ms")
    print(f"  Answer         : {resp.answer[:300]}{'...' if len(resp.answer) > 300 else ''}")

print(f"\n{'=' * 72}")
logger.success("Multi-turn complete. Memory context grew as new shopping facts were collected.")
logger.info("Long-term facts now persist in pgvector and can be reused in future sessions.")

---
## Section 11 - Observability with LangFuse

Every .chat() call creates a LangFuse trace with nested spans:

- agent_chat (trace)
- memory_recall (span)
- router (generation)
- tool_dispatch (span)
- crm / rag / web_search specialist span
- synthesiser (generation)
- memory_store (span)
- memory_distill (span)
- distill_facts (generation)

Open your LangFuse dashboard -> Traces to inspect latency, token usage, cost, and I/O for each step.

In [ ]:
from infrastructure.observability import flush

flush()  # Push pending events to LangFuse

host = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
print(f"LangFuse dashboard: {host}")
print("  Traces -> filter by tag 'agent' -> inspect the full pipeline waterfall.")
print("  Each trace: recall -> route -> dispatch -> synthesise -> store -> distill")

---
## Summary

| Component | Role | Key File |
|---|---|---|
| QueryRouter | LLM classifies intent into crm / rag / web_search / direct | agents/router.py |
| CRMTool | User profile lookups plus delivery and logistics checks | agents/tools/crm_tool.py |
| RAGTool | Product catalog retrieval, internal FAQ, and delivery knowledge | agents/tools/rag_tool.py |
| WebSearchTool | Live external search for weather, traffic, and delivery-impact news | agents/tools/web_search_tool.py |
| ShortTermStore | Ring buffer (30 turns, 24h TTL) | memory/st_store.py |
| LongTermStore | Semantic facts (pgvector, cosine KNN) | memory/lt_store.py |
| EpisodicStore | Full session snapshots (pgvector) | memory/episodic_store.py |
| ProceduralStore | Workflow steps (pgvector) | memory/procedural_store.py |
| MemoryDistiller | LLM extracts durable customer facts from turns | memory/memory_ops.py |
| MemoryRecaller | Token-budgeted hybrid recall | memory/memory_ops.py |
| LangFuse | Traces every step (latency, tokens, cost) | infrastructure/observability.py |

---

Next -> Notebook 04 replaces this imperative walkthrough with a LangGraph StateGraph, introducing typed shared state, explicit fan-out routing, and specialist sub-agents.